# Chapter 12. Deep Computer Vision Using Convolutional Neural Networks

CNNs were inspired by the visual cortex, where neurons respond to patterns within small local receptive fields. Higher-level neurons combine lower-level patterns to detect increasingly complex features.

## Convolutional Layers

Convolutional layers connect each neuron only to a local receptive field instead of every input pixel.

This local connectivity allows early layers to detect simple features, such as edges and lines, while deeper layers combine them into larger and more complex patterns.

Unlike fully connected networks, CNNs preserve the spatial structure of images instead of flattening them immediately.

This makes CNNs much more efficient for larger images, since fully connected layers would require an extremely large number of parameters.

Zero padding adds zeros around the input to preserve more of its spatial dimensions.

The `stride` controls how far the receptive field moves between positions, and a larger stride produces smaller output feature maps with lower computational cost.

### Filters

A filter, also called a kernel, is a small set of weights that slides across the input.

Each filter learns to detect a particular pattern, such as an edge, line, or texture.

Applying one filter across the whole input produces a feature map.

The feature map shows where that pattern is detected most strongly.

### Stacking Multiple Feature Maps

A convolutional layer usually contains multiple filters.

Each filter produces one feature map, so the layer output is a stack of multiple feature maps.

All neurons within the same feature map share the same filter weights and bias.

This weight sharing greatly reduces the number of parameters and allows the same feature to be detected at different locations.

For one input instance, the output shape of a convolutional layer is:

`[out_channels, output_height, output_width]`

Each output channel is one feature map, and `output_height` and `output_width` describe the spatial dimensions of that feature map.

With a batch dimension, PyTorch uses:

`[batch_size, out_channels, output_height, output_width]`

### Implementing Convolutional Layers with PyTorch

First, we load two sample RGB images, stack them into one NumPy array, convert them to a `float32` tensor, and scale the pixel values from `0–255` to `0–1`.

In [2]:
import numpy as np
import torch
from sklearn.datasets import load_sample_images

sample_images = np.stack(load_sample_images()["images"])
sample_images = torch.tensor(sample_images, dtype=torch.float32) / 255

In [4]:
sample_images.shape

torch.Size([2, 427, 640, 3])

The tensor has shape:

`[batch_size, height, width, channels]`

PyTorch convolutional layers expect image tensors in the format:

`[batch_size, channels, height, width]`

so the dimensions must be rearranged using `permute()`.

In [8]:
sample_images_permuted = sample_images.permute(0, 3, 1, 2)
sample_images_permuted.shape

torch.Size([2, 3, 427, 640])

Next, the images are center-cropped to a smaller spatial size.

In [12]:
import torchvision
import torchvision.transforms.v2 as T

cropped_images = T.CenterCrop((70, 120))(sample_images_permuted)
cropped_images.shape

torch.Size([2, 3, 70, 120])

PyTorch provides `nn.Conv2d` for 2D convolutional layers.

Here, `out_channels=32` means the layer has `32` filters, and `kernel_size=7` means that each filter has spatial dimensions `7 × 7`.

The term “2D” refers to the two spatial dimensions, height and width.

However, `nn.Conv2d` takes a 4D tensor because the batch and channel dimensions are also included.

In [14]:
import torch.nn as nn

torch.manual_seed(42)

conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7)
fmaps = conv_layer(cropped_images)

In [16]:
fmaps.shape

torch.Size([2, 32, 64, 114])

The output shape is:

`[2, 32, 64, 114]`

The number of channels increases from `3` to `32` because each of the `32` filters produces one output feature map.

The spatial dimensions shrink because the default setting is `padding=0`. With a `7 × 7` kernel and `stride=1`, the height and width each decrease by `6`.

In [18]:
conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7,
                       padding="same")
fmaps = conv_layer(cropped_images)
fmaps.shape

torch.Size([2, 32, 70, 120])

With `padding="same"`, the output spatial dimensions remain `70 × 120`.

A larger `stride` reduces the output size. For example, with `stride=2`, `padding=3`, and `kernel_size=7`, the spatial dimensions are approximately halved.

The filters and biases of an `nn.Conv2d` layer are stored in the `weight` and `bias` attributes.

In [20]:
conv_layer.weight.shape

torch.Size([32, 3, 7, 7])

In [22]:
conv_layer.bias.shape

torch.Size([32])

The weight tensor has shape:

`[out_channels, in_channels, kernel_height, kernel_width]`

The bias tensor has shape:

`[out_channels]`

The input image height and width do not appear in the filter shape because the same filter weights are shared across all spatial locations.

A convolutional layer performs a linear operation, so an activation function should normally be added after it.

Without activation functions, stacking several convolutional layers would still be equivalent to one linear convolutional operation.

The weights may also be reinitialized to match the activation function, such as using He initialization with `ReLU`.

## Pooling Layers

Pooling layers shrink feature maps to reduce computation, memory usage, and the risk of overfitting.

Unlike convolutional layers, pooling layers have no weights or biases. They simply aggregate values within each receptive field using an operation such as the maximum or mean.

Max pooling is the most common type of pooling.

For example, a `2 × 2` pooling kernel with `stride=2` keeps only the maximum value from each receptive field, reducing the height and width by about half.

Pooling is usually applied independently to each input channel, so the number of channels remains unchanged.

Max pooling provides some invariance to small translations, which can be useful for classification.

However, it is destructive because it discards many input values. For tasks such as semantic segmentation, equivariance is often more important: when the input shifts, the output should shift correspondingly.